# Difference-in-Differences: Concepts

**P@S 2026, Lecture 11 (Feb 25)**

This notebook walks through the core DiD method using the organ donation dataset from Kessler and Roth (2014), as presented in *The Effect* Chapter 18.

**What we'll do:**
1. Load the organ donation data
2. Compute DiD by hand (the 2x2 table)
3. Run the TWFE regression
4. Placebo test
5. Event study plot

## Part 1: Setup and data

In [ ]:
!pip install -q causaldata statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from causaldata import organ_donations

# Load the organ donation dataset (Kessler and Roth 2014)
df = organ_donations.load_pandas().data
print(f"Shape: {df.shape}")
print(f"States: {df['State'].nunique()}")
print(f"Quarters: {sorted(df['Quarter'].unique())}")
df.head(10)

**The data:** 27 US states, 6 quarters (Q4 2010 through Q1 2012). California switched to "active choice" organ donor registration in Q3 2011 (July 2011). The other 26 states kept their traditional opt-in systems.

**Rate** = organ donation registration rate (proportion).

In [ ]:
# Create key variables
df['California'] = (df['State'] == 'California').astype(int)
df['After'] = (df['Quarter_Num'] >= 4).astype(int)  # Q3 2011 onward
df['Treated'] = df['California'] * df['After']

# Look at California vs. others
print("California:")
print(df[df['California'] == 1][['Quarter', 'Rate', 'After']].to_string(index=False))
print("\nOther states (averages by quarter):")
print(df[df['California'] == 0].groupby('Quarter')['Rate'].mean().round(4))

## Part 2: DiD by hand (the 2x2 table)

The core of DiD is simple arithmetic with a 2x2 table.

In [ ]:
# Compute the 2x2 table
# Average rates: California vs Others, Before vs After treatment

ca_before = df[(df['California'] == 1) & (df['After'] == 0)]['Rate'].mean()
ca_after  = df[(df['California'] == 1) & (df['After'] == 1)]['Rate'].mean()
ot_before = df[(df['California'] == 0) & (df['After'] == 0)]['Rate'].mean()
ot_after  = df[(df['California'] == 0) & (df['After'] == 1)]['Rate'].mean()

print("         Before    After     Change")
print(f"CA:      {ca_before:.4f}    {ca_after:.4f}    {ca_after - ca_before:+.4f}")
print(f"Others:  {ot_before:.4f}    {ot_after:.4f}    {ot_after - ot_before:+.4f}")
print(f"\nDiD estimate: ({ca_after - ca_before:.4f}) - ({ot_after - ot_before:.4f}) = {(ca_after - ca_before) - (ot_after - ot_before):.4f}")
print(f"\nActive choice {'increased' if (ca_after - ca_before) - (ot_after - ot_before) > 0 else 'decreased'} organ donation rates by {abs((ca_after - ca_before) - (ot_after - ot_before)):.4f}")

## Part 3: Visualizing DiD

Plot the treated (California) and untreated groups over time, with the counterfactual.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# California
ca = df[df['California'] == 1].sort_values('Quarter_Num')
ax.plot(ca['Quarter_Num'], ca['Rate'], 'o-', color='#c0392b', linewidth=2, markersize=8, label='California (treated)')

# Other states average
others = df[df['California'] == 0].groupby('Quarter_Num')['Rate'].mean().reset_index()
ax.plot(others['Quarter_Num'], others['Rate'], 's-', color='#2980b9', linewidth=2, markersize=8, label='Other states (control)')

# Counterfactual: California's pre-treatment level + other states' change
ca_pre_mean = ca[ca['Quarter_Num'] < 4]['Rate'].mean()
ot_pre_mean = others[others['Quarter_Num'] < 4]['Rate'].mean()
counterfactual = others.copy()
counterfactual['Rate'] = counterfactual['Rate'] - ot_pre_mean + ca_pre_mean
cf_post = counterfactual[counterfactual['Quarter_Num'] >= 3]
ax.plot(cf_post['Quarter_Num'], cf_post['Rate'], 'o--', color='#c0392b', alpha=0.4, linewidth=2, markersize=6, label='Counterfactual (CA without treatment)')

# Treatment line
ax.axvline(x=3.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(3.55, ax.get_ylim()[1] * 0.98, 'Treatment\n(Q3 2011)', fontsize=10, color='gray', va='top')

# DiD annotation
ca_post_mean = ca[ca['Quarter_Num'] >= 4]['Rate'].mean()
cf_post_mean = counterfactual[counterfactual['Quarter_Num'] >= 4]['Rate'].mean()
ax.annotate('', xy=(5.3, ca_post_mean), xytext=(5.3, cf_post_mean),
            arrowprops=dict(arrowstyle='<->', color='#27ae60', lw=2))
ax.text(5.45, (ca_post_mean + cf_post_mean) / 2, f'DiD\n{ca_post_mean - cf_post_mean:.3f}',
        fontsize=11, color='#27ae60', fontweight='bold', va='center')

ax.set_xlabel('Quarter', fontsize=12)
ax.set_ylabel('Organ Donation Rate', fontsize=12)
ax.set_title('Difference-in-Differences: Organ Donation (Kessler & Roth 2014)', fontsize=14)
ax.set_xticks(range(1, 7))
ax.set_xticklabels(['Q4\n2010', 'Q1\n2011', 'Q2\n2011', 'Q3\n2011', 'Q4\n2011', 'Q1\n2012'])
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: TWFE regression

The two-way fixed effects regression is the standard way to estimate DiD:

$$Y_{it} = \alpha_i + \gamma_t + \beta \cdot \text{Treated}_{it} + \varepsilon_{it}$$

where $\alpha_i$ are state fixed effects and $\gamma_t$ are time fixed effects.

In [ ]:
# TWFE regression with state and quarter fixed effects
model = smf.ols('Rate ~ Treated + C(State) + C(Quarter_Num)', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['State']}
)

print("TWFE Regression: Rate ~ Treated + State FE + Time FE")
print(f"  Treated coefficient (DiD estimate): {model.params['Treated']:.4f}")
print(f"  Clustered SE:                       {model.bse['Treated']:.4f}")
print(f"  t-statistic:                        {model.tvalues['Treated']:.2f}")
print(f"  p-value:                            {model.pvalues['Treated']:.4f}")
print(f"\nInterpretation: Active choice {'increased' if model.params['Treated'] > 0 else 'decreased'} donation rates by {abs(model.params['Treated']):.4f}")
print(f"This is the Average Treatment Effect on the Treated (ATT).")

**Note:** We cluster standard errors at the state level (the level of treatment assignment). Without clustering, SEs would be too small because observations within a state are correlated over time.

## Part 5: The interaction form

An equivalent way to write DiD, making each component explicit:

In [ ]:
# Interaction form: Y = b0 + b1*California + b2*After + b3*(California*After)
model_interact = smf.ols('Rate ~ California * After', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['State']}
)

print("Interaction form: Rate ~ California + After + California:After")
print(model_interact.summary().tables[1])
print(f"\nCoefficient on California:After = {model_interact.params['California:After']:.4f}")
print("This is the DiD estimate, identical to the TWFE result.")

## Part 6: Placebo test

If parallel trends holds, there should be no "effect" before the actual treatment. We test this by picking a fake treatment date in the pre-treatment period.

In [ ]:
# Placebo test: use only pre-treatment data, fake treatment at Q2 2011
df_pre = df[df['Quarter_Num'] <= 3].copy()
df_pre['FakeAfter'] = (df_pre['Quarter_Num'] >= 3).astype(int)
df_pre['FakeTreated'] = df_pre['California'] * df_pre['FakeAfter']

placebo = smf.ols('Rate ~ FakeTreated + C(State) + C(Quarter_Num)', data=df_pre).fit(
    cov_type='cluster', cov_kwds={'groups': df_pre['State']}
)

print("Placebo test (fake treatment at Q2 2011, pre-treatment data only)")
print(f"  Fake DiD estimate: {placebo.params['FakeTreated']:.4f}")
print(f"  SE:                {placebo.bse['FakeTreated']:.4f}")
print(f"  p-value:           {placebo.pvalues['FakeTreated']:.4f}")
print(f"\nResult: {'No significant effect (good!)' if placebo.pvalues['FakeTreated'] > 0.05 else 'Significant effect (concerning!)'}")
print("A near-zero placebo estimate supports the parallel trends assumption.")

## Part 7: Event study (dynamic treatment effects)

Instead of one "before" and one "after", we estimate a separate coefficient for each time period. This produces the **event study plot**: the most important diagnostic in DiD.

In [ ]:
# Event study: interact California with each period (omit Q2 2011 = period 3 as reference)
df['RelTime'] = df['Quarter_Num'] - 3  # Center on last pre-treatment quarter

# Create period dummies interacted with California (exclude reference period 0)
for k in [-2, -1, 1, 2, 3]:
    df[f'CA_k{k}'] = ((df['RelTime'] == k) & (df['California'] == 1)).astype(int)

event_model = smf.ols(
    'Rate ~ CA_k_2 + CA_k_1 + CA_k1 + CA_k2 + CA_k3 + C(State) + C(Quarter_Num)',
    data=df.rename(columns={'CA_k-2': 'CA_k_2', 'CA_k-1': 'CA_k_1'})
).fit(cov_type='cluster', cov_kwds={'groups': df['State']})

# Extract coefficients and CIs
periods = [-2, -1, 0, 1, 2, 3]
coefs = [event_model.params.get(f'CA_k_{abs(k)}' if k < 0 else f'CA_k{k}', 0) for k in periods]
ses = [event_model.bse.get(f'CA_k_{abs(k)}' if k < 0 else f'CA_k{k}', 0) for k in periods]
# Reference period (k=0) is zero by definition
coefs[2] = 0
ses[2] = 0

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(periods, coefs, yerr=[1.96 * s for s in ses], fmt='o-', color='#2c3e50',
            linewidth=2, markersize=10, capsize=5, capthick=2)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)
ax.axvline(x=0, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.fill_between([-2.5, 0], -0.06, 0.06, alpha=0.05, color='blue')
ax.fill_between([0, 3.5], -0.06, 0.06, alpha=0.05, color='red')

ax.text(-1.2, 0.045, 'Pre-treatment\n(should be ~0)', fontsize=10, color='#2980b9', ha='center')
ax.text(2, 0.045, 'Post-treatment\n(the effect)', fontsize=10, color='#c0392b', ha='center')

ax.set_xlabel('Quarters relative to treatment (Q3 2011 = 0)', fontsize=12)
ax.set_ylabel('Estimated coefficient', fontsize=12)
ax.set_title('Event Study Plot: Organ Donation DiD', fontsize=14)
ax.set_xticks(periods)
ax.set_xticklabels(['k=-2\n(Q4 10)', 'k=-1\n(Q1 11)', 'k=0\n(Q2 11)\nref', 'k=1\n(Q3 11)', 'k=2\n(Q4 11)', 'k=3\n(Q1 12)'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nEvent study coefficients:")
for k, c, s in zip(periods, coefs, ses):
    marker = " (reference)" if k == 0 else ""
    print(f"  k={k:+d}: {c:+.4f} (SE={s:.4f}){marker}")

## Summary

| Step | What we did | Result |
|------|-------------|--------|
| **2x2 table** | Hand-computed DiD | Active choice reduced donation by ~2.2 pp |
| **TWFE regression** | State + time fixed effects, clustered SEs | Same estimate, with statistical significance |
| **Interaction form** | Explicit group x time interaction | Algebraically identical |
| **Placebo test** | Fake treatment in pre-period | Near-zero effect (parallel trends plausible) |
| **Event study** | Period-by-period coefficients | Pre-treatment flat, post-treatment negative |

**Key takeaway:** DiD is simple arithmetic (two subtractions) dressed up as a regression. The parallel trends assumption is untestable but can be made plausible with prior trends and placebo tests.